# Intake risk baseline

Compare a majority-class benchmark with L2 logistic regression for the slow-resolution label (`resolution_hours > 62.3493`). Use chronological splits: train (Oct–Dec 2023), validation (Jan–Feb 2024), and a later test period (Mar–May 2024). Model selection uses train and validation only; the one-time final test evaluation is recorded in Section 11 and must not be used for further tuning.

Model inputs are intake descriptors and calendar features. `borough` is excluded from Model 1; IDs, raw dates, status, and closure fields are not predictors.

In [1]:
import hashlib
import json
from pathlib import Path
import time

import pandas as pd
import numpy as np

from resolveiq.config import REPO_ROOT

LABELLED_DIR = REPO_ROOT / "data" / "labelled"
FEATURES_PATH = LABELLED_DIR / "heat_2023_2024_features_at_intake.json"
FEATURES_META_PATH = LABELLED_DIR / "heat_2023_2024_features_at_intake.meta.json"

LABELS_PATH = LABELLED_DIR / "heat_2023_2024_slow_resolution_labels.json"
LABELS_META_PATH = LABELLED_DIR / "heat_2023_2024_slow_resolution_labels.meta.json"

# 1. Cryptographic sidecar verification
for path, meta_path in [(FEATURES_PATH, FEATURES_META_PATH), (LABELS_PATH, LABELS_META_PATH)]:
    assert path.exists(), f"Missing file: {path}"
    assert meta_path.exists(), f"Missing sidecar: {meta_path}"
    sidecar = json.loads(meta_path.read_text(encoding="utf-8"))
    actual_sha = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual_sha == sidecar["sha256"], f"SHA-256 mismatch for {path.name}!"
    print(f"[PASS] {path.name} verified: SHA-256 {actual_sha[:16]}...")

# 2. Load into DataFrames
t0 = time.perf_counter()
df_features_raw = pd.read_json(FEATURES_PATH)
df_labels_raw = pd.read_json(LABELS_PATH)
elapsed = time.perf_counter() - t0

assert len(df_features_raw) == 235654
assert len(df_labels_raw) == 235654
assert (df_features_raw["unique_key"] == df_labels_raw["unique_key"]).all(), "Key alignment mismatch!"
print(f"[PASS] Loaded 235,654 records from both artifacts in {elapsed:.2f}s")

# 3. Test Set Quarantine: Drop test partition completely
test_count = (df_features_raw["split"] == "test").sum()
assert test_count == 47825, f"Expected 47,825 test records, found {test_count}"

dev_mask = df_features_raw["split"].isin(["train", "val"])
df_features_dev = df_features_raw[dev_mask].copy()
df_labels_dev = df_labels_raw[dev_mask].copy()

# Delete raw variables containing test data from memory
del df_features_raw, df_labels_raw

# 4. Eligibility Filter: Exclude the 162 clock anomalies
eligible_mask = df_labels_dev["eligible_for_modeling"] == True
df_features_dev = df_features_dev[eligible_mask].reset_index(drop=True)
df_labels_dev = df_labels_dev[eligible_mask].reset_index(drop=True)

# 5. Partition Verification
train_count = (df_features_dev["split"] == "train").sum()
val_count = (df_features_dev["split"] == "val").sum()

assert train_count == 106026, f"Expected 106,026 train records, got {train_count}"
assert val_count == 81641, f"Expected 81,641 val records, got {val_count}"
assert len(df_features_dev) == 187667, f"Expected 187,667 total dev records, got {len(df_features_dev)}"
assert "test" not in df_features_dev["split"].unique(), "CRITICAL: Test partition leaked into dev data!"

print("\n=== Development Dataset Summary (Test Partition Dropped) ===")
print(f"Total development records : {len(df_features_dev):,} (100% eligible for modeling)")
print(f"  - Train partition      : {train_count:,} (Oct–Dec 2023)")
print(f"  - Validation partition : {val_count:,} (Jan–Feb 2024)")
print(f"  - Test partition       : 47,825 records completely sealed and dropped from memory")
print(f"\n[PASS] Verified load and test quarantine complete.")

[PASS] heat_2023_2024_features_at_intake.json verified: SHA-256 d7895fa4663edae3...
[PASS] heat_2023_2024_slow_resolution_labels.json verified: SHA-256 ccb686ec9fdb33c0...
[PASS] Loaded 235,654 records from both artifacts in 1.95s

=== Development Dataset Summary (Test Partition Dropped) ===
Total development records : 187,667 (100% eligible for modeling)
  - Train partition      : 106,026 (Oct–Dec 2023)
  - Validation partition : 81,641 (Jan–Feb 2024)
  - Test partition       : 47,825 records completely sealed and dropped from memory

[PASS] Verified load and test quarantine complete.


## Section 2: Feature Selection & Leakage-Free Preprocessing

We define the feature set for our initial baseline model:
- **Categorical**: `["descriptor", "descriptor_2", "location_type", "open_data_channel_type", "agency"]`
- **Calendar**: `["created_hour", "created_day_of_week", "created_is_weekend"]`

**Standing Exclusions**:
- `borough` is excluded as a fairness-sensitive feature.
- `unique_key`, `created_date`, and `split` are excluded.
- Preprocessing is fitted strictly on `train` to prevent data snooping leakage into `val`.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Define explicit feature columns
CATEGORICAL_FEATURES = [
    "descriptor",
    "descriptor_2",
    "location_type",
    "open_data_channel_type",
    "agency",
]

NUMERIC_FEATURES = [
    "created_hour",
    "created_day_of_week",
    "created_is_weekend",
]

MODEL_FEATURE_COLS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# Enforce exclusions
assert "borough" not in MODEL_FEATURE_COLS, "CRITICAL: borough must not enter the first baseline!"
assert "created_date" not in MODEL_FEATURE_COLS
assert "unique_key" not in MODEL_FEATURE_COLS

# 2. Partition into X and y for train and val
train_mask = df_features_dev["split"] == "train"
val_mask = df_features_dev["split"] == "val"

X_train_raw = df_features_dev.loc[train_mask, MODEL_FEATURE_COLS].copy()
y_train = df_labels_dev.loc[train_mask, "slow_resolution_label"].astype(int).values

X_val_raw = df_features_dev.loc[val_mask, MODEL_FEATURE_COLS].copy()
y_val = df_labels_dev.loc[val_mask, "slow_resolution_label"].astype(int).values

# Impute missing categorical values with explicit sentinel
for col in CATEGORICAL_FEATURES:
    X_train_raw[col] = X_train_raw[col].fillna("MISSING").astype(str)
    X_val_raw[col] = X_val_raw[col].fillna("MISSING").astype(str)

# 3. Build scikit-learn Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ("num", StandardScaler(), NUMERIC_FEATURES),
    ],
    remainder="drop",
)

# Fit strictly on train, then transform both train and val
X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)

# Extract encoded feature names for interpretability
cat_feature_names = list(preprocessor.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
encoded_feature_names = cat_feature_names + NUMERIC_FEATURES

# 4. Verification Assertions
assert X_train.shape[0] == 106026
assert X_val.shape[0] == 81641
assert X_train.shape[1] == X_val.shape[1] == len(encoded_feature_names)
assert len(y_train) == 106026
assert len(y_val) == 81641

print("[PASS] Preprocessing pipeline fitted strictly on train.")
print(f"X_train shape           : {X_train.shape} (106,026 rows, {X_train.shape[1]} features)")
print(f"X_val shape             : {X_val.shape} (81,641 rows, {X_val.shape[1]} features)")
print(f"y_train positive class  : {y_train.sum():,} ({y_train.mean():.1%})")
print(f"y_val positive class    : {y_val.sum():,} ({y_val.mean():.1%})")
print(f"\nEncoded feature count   : {len(encoded_feature_names)}")
print(f"Sample encoded features : {encoded_feature_names[:6]} ... {encoded_feature_names[-3:]}")

[PASS] Preprocessing pipeline fitted strictly on train.
X_train shape           : (106026, 21) (106,026 rows, 21 features)
X_val shape             : (81641, 21) (81,641 rows, 21 features)
y_train positive class  : 27,831 (26.2%)
y_val positive class    : 12,185 (14.9%)

Encoded feature count   : 21
Sample encoded features : ['descriptor_APARTMENT ONLY', 'descriptor_Apartment Only', 'descriptor_ENTIRE BUILDING', 'descriptor_Entire Building', 'descriptor_Water Supply', 'descriptor_2_NO HEAT'] ... ['created_hour', 'created_day_of_week', 'created_is_weekend']


## Section 3: Majority-Class Baseline (Benchmark 0)

We evaluate a featureless dummy baseline on `val` to establish the minimum operational floor:
1. **Majority-class vote** ($\hat{y} = 0$ everywhere): Tests naive accuracy vs. recall.
2. **Constant-prevalence prior** ($\hat{p} = \bar{y}_{\text{train}} \approx 0.2625$): Sets the baseline for PR-AUC ($0.1493$) and probability calibration.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# 1. Fit Dummy Classifier on Train (prior probability strategy)
dummy_clf = DummyClassifier(strategy="prior")
dummy_clf.fit(X_train, y_train)

# 2. Predict on Validation
y_val_pred_dummy = dummy_clf.predict(X_val)  # Predicts 0 for all rows
y_val_prob_dummy = dummy_clf.predict_proba(X_val)[:, 1]  # Constant 0.2625

# 3. Compute Baseline Metrics
dummy_metrics = {
    "Model": "Majority-Class Prior",
    "Accuracy": accuracy_score(y_val, y_val_pred_dummy),
    "Precision": precision_score(y_val, y_val_pred_dummy, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred_dummy, zero_division=0),
    "F1-Score": f1_score(y_val, y_val_pred_dummy, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob_dummy),
    "PR-AUC (Avg Precision)": average_precision_score(y_val, y_val_prob_dummy),
    "Brier Score": brier_score_loss(y_val, y_val_prob_dummy),
    "Log Loss": log_loss(y_val, y_val_prob_dummy),
}

df_benchmark = pd.DataFrame([dummy_metrics])

print("=== Majority-Class Baseline on Validation (Peak Winter) ===")
for metric, val in dummy_metrics.items():
    if metric == "Model":
        continue
    print(f"{metric:<24} : {val:.4f}")

print("\nKey Takeaways:")
print(f"1. Naive accuracy is high ({dummy_metrics['Accuracy']:.1%}), but catches ZERO slow requests (Recall = 0.0%).")
print(f"2. PR-AUC baseline is exactly the validation positive prevalence ({dummy_metrics['PR-AUC (Avg Precision)']:.1%}).")
print(f"3. ROC-AUC is exactly {dummy_metrics['ROC-AUC']:.4f} (random guessing rank order).")

=== Majority-Class Baseline on Validation (Peak Winter) ===
Accuracy                 : 0.8507
Precision                : 0.0000
Recall                   : 0.0000
F1-Score                 : 0.0000
ROC-AUC                  : 0.5000
PR-AUC (Avg Precision)   : 0.1493
Brier Score              : 0.1398
Log Loss                 : 0.4587

Key Takeaways:
1. Naive accuracy is high (85.1%), but catches ZERO slow requests (Recall = 0.0%).
2. PR-AUC baseline is exactly the validation positive prevalence (14.9%).
3. ROC-AUC is exactly 0.5000 (random guessing rank order).


## Section 4: Simple Tabular Linear Model (Logistic Regression)

We fit an $L_2$-regularized Logistic Regression model on `X_train` using only the approved intake features (excluding `borough`).

We evaluate whether intake complaint shapes and timing offer genuine discriminatory power over the majority baseline on the chronological validation split.

In [4]:
from sklearn.linear_model import LogisticRegression

# 1. Fit Logistic Regression strictly on Train
t0 = time.perf_counter()
lr_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=42,
)
lr_model.fit(X_train, y_train)
fit_time = time.perf_counter() - t0

# 2. Predict Probabilities and Classes on Validation
y_val_prob_lr = lr_model.predict_proba(X_val)[:, 1]
y_val_pred_lr = (y_val_prob_lr >= 0.5).astype(int)

# 3. Compute Metrics
lr_metrics = {
    "Model": "Logistic Regression (No Borough)",
    "Accuracy": accuracy_score(y_val, y_val_pred_lr),
    "Precision": precision_score(y_val, y_val_pred_lr, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred_lr, zero_division=0),
    "F1-Score": f1_score(y_val, y_val_pred_lr, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob_lr),
    "PR-AUC (Avg Precision)": average_precision_score(y_val, y_val_prob_lr),
    "Brier Score": brier_score_loss(y_val, y_val_prob_lr),
    "Log Loss": log_loss(y_val, y_val_prob_lr),
}

# 4. Side-by-Side Benchmark Comparison
df_comparison = pd.DataFrame([dummy_metrics, lr_metrics]).set_index("Model")

print(f"[PASS] Logistic Regression fitted in {fit_time:.2f}s\n")
print("=== Side-by-Side Model Comparison on Validation Partition ===")
print(df_comparison.round(4).to_string())

# Compute relative lifts
roc_lift = lr_metrics["ROC-AUC"] - dummy_metrics["ROC-AUC"]
prauc_lift = lr_metrics["PR-AUC (Avg Precision)"] - dummy_metrics["PR-AUC (Avg Precision)"]
brier_reduction = dummy_metrics["Brier Score"] - lr_metrics["Brier Score"]

print(f"\nObserved Lift over Baseline:")
print(f"  ROC-AUC gain      : +{roc_lift:.4f} (from 0.5000 to {lr_metrics['ROC-AUC']:.4f})")
print(f"  PR-AUC gain       : +{prauc_lift:.4f} (from {dummy_metrics['PR-AUC (Avg Precision)']:.4f} to {lr_metrics['PR-AUC (Avg Precision)']:.4f})")
print(f"  Brier score drop  : -{brier_reduction:.4f} (lower is better, from {dummy_metrics['Brier Score']:.4f} to {lr_metrics['Brier Score']:.4f})")

[PASS] Logistic Regression fitted in 0.09s

=== Side-by-Side Model Comparison on Validation Partition ===
                                  Accuracy  Precision  Recall  F1-Score  ROC-AUC  PR-AUC (Avg Precision)  Brier Score  Log Loss
Model                                                                                                                          
Majority-Class Prior                0.8507        0.0     0.0       0.0   0.5000                  0.1493       0.1398    0.4587
Logistic Regression (No Borough)    0.8507        0.0     0.0       0.0   0.6385                  0.2202       0.1375    0.4459

Observed Lift over Baseline:
  ROC-AUC gain      : +0.1385 (from 0.5000 to 0.6385)
  PR-AUC gain       : +0.0709 (from 0.1493 to 0.2202)
  Brier score drop  : -0.0023 (lower is better, from 0.1398 to 0.1375)


c:\Users\mejdi\Documents\resolveiq\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## Section 5: Decision Thresholds & Operational Triage Analysis

In operational triage with an imbalanced base rate ($14.9\%$), an arbitrary $0.50$ threshold is non-operational. 

We evaluate:
1. **Threshold Sweep**: Workload, Precision, and Recall across operational cutoffs $\hat{p} \in [0.10, 0.35]$.
2. **Decile Risk Ranking**: Concentrating inspection capacity on the highest-risk requests.

In [5]:
# 1. Operational Threshold Sweep
thresholds = [0.10, 0.15, 0.18, 0.20, 0.22, 0.25, 0.30]
sweep_rows = []

for t in thresholds:
    pred_t = (y_val_prob_lr >= t).astype(int)
    flagged = pred_t.sum()
    prec = precision_score(y_val, pred_t, zero_division=0)
    rec = recall_score(y_val, pred_t, zero_division=0)
    f1 = f1_score(y_val, pred_t, zero_division=0)
    
    sweep_rows.append({
        "Threshold (p >= t)": f"{t:.2f}",
        "Tickets Flagged": f"{flagged:,}",
        "Workload %": f"{flagged / len(y_val):.1%}",
        "Precision": f"{prec:.1%}",
        "Recall": f"{rec:.1%}",
        "F1-Score": f"{f1:.4f}",
    })

df_sweep = pd.DataFrame(sweep_rows)
print("=== 1. Operational Decision Threshold Sweep on Validation ===")
print(df_sweep.to_string(index=False))

# 2. Decile Lift Analysis (Sorting by Predicted Risk)
df_decile = pd.DataFrame({
    "y_true": y_val,
    "y_prob": y_val_prob_lr,
})

# Bin into 10 risk deciles (Decile 1 = highest risk, Decile 10 = lowest)
df_decile["decile"] = pd.qcut(df_decile["y_prob"], q=10, labels=False, duplicates="drop")
# Invert so Decile 1 is top risk
df_decile["decile"] = df_decile["decile"].max() - df_decile["decile"] + 1

decile_summary = []
base_rate = y_val.mean()
total_slow_val = y_val.sum()

cum_slow = 0
for d in sorted(df_decile["decile"].unique()):
    subset = df_decile[df_decile["decile"] == d]
    n_tickets = len(subset)
    slow_count = subset["y_true"].sum()
    slow_rate = slow_count / n_tickets
    lift = slow_rate / base_rate
    cum_slow += slow_count
    cum_recall = cum_slow / total_slow_val
    
    decile_summary.append({
        "Risk Decile": f"Decile {d}",
        "Cases": f"{n_tickets:,}",
        "Mean Pred Risk": f"{subset['y_prob'].mean():.1%}",
        "Actual Slow %": f"{slow_rate:.1%}",
        "Lift over Prior": f"{lift:.2f}x",
        "Cumulative Recall": f"{cum_recall:.1%}",
    })

df_decile_table = pd.DataFrame(decile_summary)
print("\n=== 2. Risk Decile Lift Table (Highest to Lowest Risk) ===")
print(df_decile_table.to_string(index=False))

=== 1. Operational Decision Threshold Sweep on Validation ===
Threshold (p >= t) Tickets Flagged Workload % Precision Recall F1-Score
              0.10          81,248      99.5%     15.0%  99.8%   0.2604
              0.15          69,206      84.8%     16.2%  91.8%   0.2750
              0.18          60,634      74.3%     17.6%  87.5%   0.2929
              0.20          56,588      69.3%     18.4%  85.2%   0.3020
              0.22          53,177      65.1%     18.8%  82.1%   0.3061
              0.25          44,712      54.8%     20.2%  74.0%   0.3170
              0.30          31,165      38.2%     21.1%  54.0%   0.3036

=== 2. Risk Decile Lift Table (Highest to Lowest Risk) ===
Risk Decile Cases Mean Pred Risk Actual Slow % Lift over Prior Cumulative Recall
   Decile 1 8,160          43.1%         26.2%           1.75x             17.5%
   Decile 2 8,152          37.0%         22.0%           1.48x             32.3%
   Decile 3 8,125          33.8%         17.1%           1.

## Section 6: Feature Importance & Operational Interpretability

We extract the model coefficients and convert them into **Odds Ratios** ($\exp(w_j)$) to identify which complaint types, intake channels, and calendar factors drive slow-resolution risk.

In [6]:
# 1. Extract coefficients and calculate Odds Ratios
coefficients = lr_model.coef_[0]
odds_ratios = np.exp(coefficients)

df_importance = pd.DataFrame({
    "Feature": encoded_feature_names,
    "Coefficient (log-odds)": coefficients,
    "Odds Ratio": odds_ratios,
}).sort_values(by="Coefficient (log-odds)", ascending=False)

print("=== 1. Top Risk-Increasing Intake Features (Odds Ratio > 1.0) ===")
print("Features that significantly increase the odds of slow resolution (>62.35h):\n")
top_risk = df_importance[df_importance["Coefficient (log-odds)"] > 0].head(8)
for _, row in top_risk.iterrows():
    pct_increase = (row["Odds Ratio"] - 1.0) * 100
    print(f"  {row['Feature']:<42} : OR = {row['Odds Ratio']:.3f} (+{pct_increase:.1f}% odds of delay)")

print("\n=== 2. Top Protective Intake Features (Odds Ratio < 1.0) ===")
print("Features that correlate with faster resolution:\n")
top_protective = df_importance[df_importance["Coefficient (log-odds)"] < 0].tail(8).iloc[::-1]
for _, row in top_protective.iterrows():
    pct_decrease = (1.0 - row["Odds Ratio"]) * 100
    print(f"  {row['Feature']:<42} : OR = {row['Odds Ratio']:.3f} (-{pct_decrease:.1f}% odds of delay)")

print("\n=== 3. Temporal Factors (Standardized) ===")
temporal_df = df_importance[df_importance["Feature"].isin(NUMERIC_FEATURES)]
for _, row in temporal_df.iterrows():
    print(f"  {row['Feature']:<42} : Coef = {row['Coefficient (log-odds)']:+.4f} | OR = {row['Odds Ratio']:.3f}")

=== 1. Top Risk-Increasing Intake Features (Odds Ratio > 1.0) ===
Features that significantly increase the odds of slow resolution (>62.35h):

  created_day_of_week                        : OR = 2.254 (+125.4% odds of delay)
  descriptor_2_NO HEAT                       : OR = 1.057 (+5.7% odds of delay)
  descriptor_Water Supply                    : OR = 1.003 (+0.3% odds of delay)

=== 2. Top Protective Intake Features (Odds Ratio < 1.0) ===
Features that correlate with faster resolution:

  created_is_weekend                         : OR = 0.618 (-38.2% odds of delay)
  agency_HPD                                 : OR = 0.690 (-31.0% odds of delay)
  open_data_channel_type_MOBILE              : OR = 0.830 (-17.0% odds of delay)
  descriptor_Entire Building                 : OR = 0.839 (-16.1% odds of delay)
  descriptor_2_NO HOT WATER                  : OR = 0.857 (-14.3% odds of delay)
  open_data_channel_type_ONLINE              : OR = 0.858 (-14.2% odds of delay)
  location_type_RE

## Section 7: Borough Fairness & Error Rate Audit (Validation Partition)

We evaluate the baseline model (trained without `borough`) across each of the 5 NYC boroughs on `val` to check for calibration gaps and error rate disparities at operational threshold $t = 0.25$.

In [7]:
# 1. Assemble validation evaluation DataFrame with borough labels
df_val_audit = pd.DataFrame({
    "borough": df_features_dev.loc[val_mask, "borough"].values,
    "y_true": y_val,
    "y_prob": y_val_prob_lr,
})

OPERATIONAL_THRESHOLD = 0.25
df_val_audit["y_pred"] = (df_val_audit["y_prob"] >= OPERATIONAL_THRESHOLD).astype(int)

# 2. Compute Disaggregated Metrics per Borough
borough_rows = []
for b in sorted(df_val_audit["borough"].unique()):
    sub = df_val_audit[df_val_audit["borough"] == b]
    n = len(sub)
    actual_pos = sub["y_true"].sum()
    actual_rate = actual_pos / n
    mean_prob = sub["y_prob"].mean()
    calib_gap = mean_prob - actual_rate  # Positive = overpredicts delay, Negative = underpredicts
    
    flagged = sub["y_pred"].sum()
    selection_rate = flagged / n
    
    tp = ((sub["y_true"] == 1) & (sub["y_pred"] == 1)).sum()
    fp = ((sub["y_true"] == 0) & (sub["y_pred"] == 1)).sum()
    tn = ((sub["y_true"] == 0) & (sub["y_pred"] == 0)).sum()
    fn = ((sub["y_true"] == 1) & (sub["y_pred"] == 0)).sum()
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    roc = roc_auc_score(sub["y_true"], sub["y_prob"])
    pr = average_precision_score(sub["y_true"], sub["y_prob"])
    
    borough_rows.append({
        "Borough": b,
        "Cases": f"{n:,}",
        "Actual Slow %": f"{actual_rate:.1%}",
        "Mean Pred %": f"{mean_prob:.1%}",
        "Calib Gap": f"{calib_gap:+.1%}",
        "Flagged %": f"{selection_rate:.1%}",
        "FPR": f"{fpr:.1%}",
        "FNR": f"{fnr:.1%}",
        "Precision": f"{prec:.1%}",
        "Recall": f"{rec:.1%}",
        "ROC-AUC": f"{roc:.4f}",
        "PR-AUC": f"{pr:.4f}",
    })

df_borough_audit = pd.DataFrame(borough_rows)

print(f"=== Baseline Model (No Borough) Disaggregated by Borough on Validation ===")
print(f"Operational Decision Threshold : p >= {OPERATIONAL_THRESHOLD:.2f}\n")
print(df_borough_audit.to_string(index=False))

print("\nKey Audit Observations:")
print("1. Calibration Gap: Because the model is blind to borough, its mean predicted probability (~26%) reflects the citywide training prior.")
print("2. In the Bronx (actual slow rate 8.3%), the model substantially over-predicts delay, yielding a high False Positive Rate.")
print("3. In Manhattan (actual slow rate 24.8%), the model is closer to true delay risk.")

=== Baseline Model (No Borough) Disaggregated by Borough on Validation ===
Operational Decision Threshold : p >= 0.25

      Borough  Cases Actual Slow % Mean Pred % Calib Gap Flagged %   FPR   FNR Precision Recall ROC-AUC PR-AUC
        BRONX 30,139          8.3%       26.1%    +17.8%     53.4% 50.8% 18.2%     12.7%  81.8%  0.6953 0.1630
     BROOKLYN 20,915         16.1%       26.5%    +10.5%     55.5% 52.4% 28.5%     20.7%  71.5%  0.6131 0.2196
    MANHATTAN 17,604         24.8%       26.3%     +1.5%     55.1% 49.6% 28.2%     32.3%  71.8%  0.6356 0.3382
       QUEENS 12,117         14.6%       26.7%    +12.1%     56.3% 53.4% 27.2%     18.9%  72.8%  0.6197 0.1989
STATEN ISLAND    866         23.3%       27.2%     +3.9%     57.9% 51.7% 21.8%     31.5%  78.2%  0.6867 0.3862

Key Audit Observations:
1. Calibration Gap: Because the model is blind to borough, its mean predicted probability (~26%) reflects the citywide training prior.
2. In the Bronx (actual slow rate 8.3%), the model subs

## Section 8: Model Comparison (With `borough`) & Non-Causal Interpretations

We fit a second Logistic Regression model that includes `borough` as a categorical feature to compare:
1. Discrimination (ROC-AUC, PR-AUC) and calibration (Brier score) against the No-Borough baseline.
2. Calibration gap improvements across the 5 boroughs.
3. Feature associations using non-causal language and standard-deviation unit scaling for calendar features.

In [8]:
# 1. Build Preprocessor with Borough Included
CAT_COLS_WITH_BOROUGH = ["borough"] + CATEGORICAL_FEATURES

X_train_raw_b = df_features_dev.loc[train_mask, ["borough"] + MODEL_FEATURE_COLS].copy()
X_val_raw_b = df_features_dev.loc[val_mask, ["borough"] + MODEL_FEATURE_COLS].copy()

for col in CAT_COLS_WITH_BOROUGH:
    X_train_raw_b[col] = X_train_raw_b[col].fillna("MISSING").astype(str)
    X_val_raw_b[col] = X_val_raw_b[col].fillna("MISSING").astype(str)

preprocessor_b = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS_WITH_BOROUGH),
        ("num", StandardScaler(), NUMERIC_FEATURES),
    ],
    remainder="drop",
)

X_train_b = preprocessor_b.fit_transform(X_train_raw_b)
X_val_b = preprocessor_b.transform(X_val_raw_b)

cat_names_b = list(preprocessor_b.named_transformers_["cat"].get_feature_names_out(CAT_COLS_WITH_BOROUGH))
feature_names_b = cat_names_b + NUMERIC_FEATURES

# 2. Fit Logistic Regression with Borough
lr_model_b = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=42)
lr_model_b.fit(X_train_b, y_train)

y_val_prob_b = lr_model_b.predict_proba(X_val_b)[:, 1]
y_val_pred_b = (y_val_prob_b >= OPERATIONAL_THRESHOLD).astype(int)

# 3. Overall Model Comparison Table
lr_metrics_b = {
    "Model": "Logistic Regression (WITH Borough)",
    "Accuracy": accuracy_score(y_val, (y_val_prob_b >= 0.5).astype(int)),
    "Precision": precision_score(y_val, y_val_pred_b, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred_b, zero_division=0),
    "F1-Score": f1_score(y_val, y_val_pred_b, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob_b),
    "PR-AUC (Avg Precision)": average_precision_score(y_val, y_val_prob_b),
    "Brier Score": brier_score_loss(y_val, y_val_prob_b),
    "Log Loss": log_loss(y_val, y_val_prob_b),
}

df_model_comp = pd.DataFrame([dummy_metrics, lr_metrics, lr_metrics_b]).set_index("Model")
print("=== 1. Overall Validation Metrics Comparison ===")
print(df_model_comp.round(4).to_string())

# 4. Calibration Gap Comparison across Boroughs
calib_compare = []
for b in sorted(df_val_audit["borough"].unique()):
    idx = df_val_audit["borough"] == b
    actual = y_val[idx].mean()
    prob_no_b = y_val_prob_lr[idx].mean()
    prob_with_b = y_val_prob_b[idx].mean()
    
    calib_compare.append({
        "Borough": b,
        "Actual Slow %": f"{actual:.1%}",
        "No-Borough Pred": f"{prob_no_b:.1%}",
        "No-Borough Gap": f"{prob_no_b - actual:+.1%}",
        "With-Borough Pred": f"{prob_with_b:.1%}",
        "With-Borough Gap": f"{prob_with_b - actual:+.1%}",
    })

print("\n=== 2. Calibration Gap Comparison by Borough ===")
print(pd.DataFrame(calib_compare).to_string(index=False))

# 5. Revised Non-Causal Odds Ratio Interpretations
df_coef_b = pd.DataFrame({
    "Feature": feature_names_b,
    "Coefficient": lr_model_b.coef_[0],
    "Odds Ratio": np.exp(lr_model_b.coef_[0]),
}).sort_values(by="Coefficient", ascending=False)

print("\n=== 3. Revised Feature Odds Ratios (Non-Causal Language) ===")
print("\nA. Borough Indicator Associations (relative to baseline):")
for _, r in df_coef_b[df_coef_b["Feature"].str.startswith("borough_")].iterrows():
    diff = (r["Odds Ratio"] - 1.0) * 100
    direction = "higher" if diff > 0 else "lower"
    print(f"  {r['Feature']:<30} : OR = {r['Odds Ratio']:.3f} (associated with {abs(diff):.1f}% {direction} odds of delay)")

print("\nB. Standardized Numeric Factors (Per 1-Standard-Deviation Increase):")
std_dow = X_train_raw["created_day_of_week"].std()
std_hour = X_train_raw["created_hour"].std()

for _, r in df_coef_b[df_coef_b["Feature"].isin(NUMERIC_FEATURES)].iterrows():
    feat = r["Feature"]
    diff = (r["Odds Ratio"] - 1.0) * 100
    direction = "higher" if diff > 0 else "lower"
    scale_desc = f"1 SD (~{std_dow:.1f} days)" if feat == "created_day_of_week" else (f"1 SD (~{std_hour:.1f} hours)" if feat == "created_hour" else "1 SD")
    print(f"  {feat:<25} : OR = {r['Odds Ratio']:.3f} (a {scale_desc} increase is associated with {abs(diff):.1f}% {direction} odds of delay)")

=== 1. Overall Validation Metrics Comparison ===
                                    Accuracy  Precision  Recall  F1-Score  ROC-AUC  PR-AUC (Avg Precision)  Brier Score  Log Loss
Model                                                                                                                            
Majority-Class Prior                  0.8507      0.000   0.000    0.0000   0.5000                  0.1493       0.1398    0.4587
Logistic Regression (No Borough)      0.8507      0.000   0.000    0.0000   0.6385                  0.2202       0.1375    0.4459
Logistic Regression (WITH Borough)    0.8405      0.209   0.694    0.3212   0.6732                  0.2609       0.1345    0.4363

=== 2. Calibration Gap Comparison by Borough ===
      Borough Actual Slow % No-Borough Pred No-Borough Gap With-Borough Pred With-Borough Gap
        BRONX          8.3%           26.1%         +17.8%             22.2%           +13.9%
     BROOKLYN         16.1%           26.5%         +10.5%     

## Section 9: Disaggregated Model Comparison & Reference Contrasts ($t = 0.25$)

We directly compare Model 1 (No Borough) and Model 2 (WITH Borough) across all 5 boroughs on `val` using the operational threshold $t = 0.25$. 

We also report borough odds ratios as explicit pairwise contrasts against **Brooklyn as the reference borough**.

In [10]:
# 1. Compute Side-by-Side Disaggregated Metrics at t = 0.25
OPERATIONAL_THRESHOLD = 0.25

comp_rows = []
for b in sorted(df_val_audit["borough"].unique()):
    mask_b = df_val_audit["borough"] == b
    y_sub = y_val[mask_b]
    n_sub = len(y_sub)
    actual_rate = y_sub.mean()
    
    # Model 1: No Borough
    p1 = y_val_prob_lr[mask_b]
    pred1 = (p1 >= OPERATIONAL_THRESHOLD).astype(int)
    
    sel1 = pred1.mean()
    fp1 = ((y_sub == 0) & (pred1 == 1)).sum()
    fn1 = ((y_sub == 1) & (pred1 == 0)).sum()
    tp1 = ((y_sub == 1) & (pred1 == 1)).sum()
    tn1 = ((y_sub == 0) & (pred1 == 0)).sum()
    
    fpr1 = fp1 / (fp1 + tn1) if (fp1 + tn1) > 0 else 0.0
    fnr1 = fn1 / (fn1 + tp1) if (fn1 + tp1) > 0 else 0.0
    prec1 = tp1 / (tp1 + fp1) if (tp1 + fp1) > 0 else 0.0
    rec1 = tp1 / (tp1 + fn1) if (tp1 + fn1) > 0 else 0.0
    gap1 = p1.mean() - actual_rate
    
    # Model 2: With Borough
    p2 = y_val_prob_b[mask_b]
    pred2 = (p2 >= OPERATIONAL_THRESHOLD).astype(int)
    
    sel2 = pred2.mean()
    fp2 = ((y_sub == 0) & (pred2 == 1)).sum()
    fn2 = ((y_sub == 1) & (pred2 == 0)).sum()
    tp2 = ((y_sub == 1) & (pred2 == 1)).sum()
    tn2 = ((y_sub == 0) & (pred2 == 0)).sum()
    
    fpr2 = fp2 / (fp2 + tn2) if (fp2 + tn2) > 0 else 0.0
    fnr2 = fn2 / (fn2 + tp2) if (fn2 + tp2) > 0 else 0.0
    prec2 = tp2 / (tp2 + fp2) if (tp2 + fp2) > 0 else 0.0
    rec2 = tp2 / (tp2 + fn2) if (tp2 + fn2) > 0 else 0.0
    gap2 = p2.mean() - actual_rate
    
    comp_rows.append({
        "Borough": b,
        "Cases": f"{n_sub:,}",
        "Actual Slow %": f"{actual_rate:.1%}",
        # No Borough
        "M1 Sel %": f"{sel1:.1%}",
        "M1 FPR": f"{fpr1:.1%}",
        "M1 FNR": f"{fnr1:.1%}",
        "M1 Prec": f"{prec1:.1%}",
        "M1 Rec": f"{rec1:.1%}",
        "M1 Gap": f"{gap1:+.1%}",
        # With Borough
        "M2 Sel %": f"{sel2:.1%}",
        "M2 FPR": f"{fpr2:.1%}",
        "M2 FNR": f"{fn2:.1%}",
        "M2 Prec": f"{prec2:.1%}",
        "M2 Rec": f"{rec2:.1%}",
        "M2 Gap": f"{gap2:+.1%}",
    })

df_side_by_side = pd.DataFrame(comp_rows)

print("=== Disaggregated Comparison at Threshold t = 0.25 (Validation Partition) ===")
print("M1 = Logistic Regression (No Borough) | M2 = Logistic Regression (WITH Borough)\n")
print(df_side_by_side.to_string(index=False))

# 2. Reference-Borough Contrasts (Brooklyn = Reference)
REFERENCE_BOROUGH = "borough_BROOKLYN"
b_coef_dict = dict(zip(feature_names_b, lr_model_b.coef_[0]))
w_ref = b_coef_dict[REFERENCE_BOROUGH]

print(f"\n=== Borough Odds Ratios Relative to {REFERENCE_BOROUGH.replace('borough_', '')} (Reference) ===")
print("Holding complaint descriptor, channel, location type, and timing constant:\n")

for b_col in sorted([k for k in b_coef_dict if k.startswith("borough_")]):
    b_name = b_col.replace("borough_", "")
    if b_col == REFERENCE_BOROUGH:
        print(f"  {b_name:<15} : Reference Category (OR = 1.000)")
    else:
        log_odds_contrast = b_coef_dict[b_col] - w_ref
        or_contrast = np.exp(log_odds_contrast)
        pct_diff = (or_contrast - 1.0) * 100
        direction = "higher" if pct_diff > 0 else "lower"
        print(f"  {b_name:<15} : OR = {or_contrast:.3f} ({abs(pct_diff):.1f}% {direction} odds of delay vs Brooklyn)")

=== Disaggregated Comparison at Threshold t = 0.25 (Validation Partition) ===
M1 = Logistic Regression (No Borough) | M2 = Logistic Regression (WITH Borough)

      Borough  Cases Actual Slow % M1 Sel % M1 FPR M1 FNR M1 Prec M1 Rec M1 Gap M2 Sel % M2 FPR    M2 FNR M2 Prec M2 Rec M2 Gap
        BRONX 30,139          8.3%    53.4%  50.8%  18.2%   12.7%  81.8% +17.8%    39.0%  37.0%  97200.0%   13.0%  61.1% +13.9%
     BROOKLYN 20,915         16.1%    55.5%  52.4%  28.5%   20.7%  71.5% +10.5%    31.1%  28.7% 188200.0%   22.7%  44.0%  +4.6%
    MANHATTAN 17,604         24.8%    55.1%  49.6%  28.2%   32.3%  71.8%  +1.5%    79.6%  76.4%  47600.0%   27.7%  89.1% +12.3%
       QUEENS 12,117         14.6%    56.3%  53.4%  27.2%   18.9%  72.8% +12.1%    62.3%  59.5%  38100.0%   18.4%  78.4% +14.1%
STATEN ISLAND    866         23.3%    57.9%  51.7%  21.8%   31.5%  78.2%  +3.9%    73.2%  67.8%   1800.0%   29.0%  91.1% +10.1%

=== Borough Odds Ratios Relative to BROOKLYN (Reference) ===
Holding com

## Section 10: Fixed Review-Capacity Evaluation (Top 10%, 20%, 30%)

We compare Model 1 (No Borough) and Model 2 (WITH Borough) under operational constraints where triage review is capped at the top 10%, 20%, and 30% of incoming volume.

In [11]:
# 1. Helper function to evaluate fixed capacity
def evaluate_capacity(y_true, y_prob, borough_series, capacity_pct):
    n_total = len(y_true)
    k = int(round(capacity_pct * n_total))
    
    # Select top-k highest risk cases
    top_indices = np.argsort(-y_prob)[:k]
    y_pred = np.zeros(n_total, dtype=int)
    y_pred[top_indices] = 1
    
    # Overall metrics
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    
    overall = {
        "Capacity": f"Top {int(capacity_pct * 100)}%",
        "Flagged": f"{k:,}",
        "Precision": f"{prec:.1%}",
        "Recall": f"{rec:.1%}",
        "FPR": f"{fpr:.1%}",
    }
    
    # Per-borough metrics
    borough_records = []
    for b in sorted(borough_series.unique()):
        mask_b = borough_series == b
        sub_y = y_true[mask_b]
        sub_pred = y_pred[mask_b]
        n_b = len(sub_y)
        
        tp_b = ((sub_y == 1) & (sub_pred == 1)).sum()
        fp_b = ((sub_y == 0) & (sub_pred == 1)).sum()
        tn_b = ((sub_y == 0) & (sub_pred == 0)).sum()
        fn_b = ((sub_y == 1) & (sub_pred == 0)).sum()
        
        prec_b = tp_b / (tp_b + fp_b) if (tp_b + fp_b) > 0 else 0.0
        rec_b = tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0.0
        fpr_b = fp_b / (fp_b + tn_b) if (fp_b + tn_b) > 0 else 0.0
        sel_b = sub_pred.mean()
        
        borough_records.append({
            "Capacity": f"Top {int(capacity_pct * 100)}%",
            "Borough": b,
            "Cases": f"{n_b:,}",
            "Share Flagged": f"{sel_b:.1%}",
            "Precision": f"{prec_b:.1%}",
            "Recall": f"{rec_b:.1%}",
            "FPR": f"{fpr_b:.1%}",
        })
        
    return overall, borough_records

# 2. Run for Capacities 10%, 20%, 30% for Both Models
boroughs_val = df_features_dev.loc[val_mask, "borough"].values
capacities = [0.10, 0.20, 0.30]

m1_overall, m1_boroughs = [], []
m2_overall, m2_boroughs = [], []

for cap in capacities:
    # Model 1
    o1, b1 = evaluate_capacity(y_val, y_val_prob_lr, boroughs_val, cap)
    m1_overall.append(o1)
    m1_boroughs.extend(b1)
    
    # Model 2
    o2, b2 = evaluate_capacity(y_val, y_val_prob_b, boroughs_val, cap)
    m2_overall.append(o2)
    m2_boroughs.extend(b2)

print("=== 1. Overall Comparison Across Fixed Review Capacities ===")
df_m1_ov = pd.DataFrame(m1_overall).set_index("Capacity")
df_m2_ov = pd.DataFrame(m2_overall).set_index("Capacity")
ov_merged = df_m1_ov.join(df_m2_ov, lsuffix=" (M1 No Borough)", rsuffix=" (M2 With Borough)")
print(ov_merged.to_string())

print("\n=== 2. Disaggregated Borough Comparison (Model 1: No Borough) ===")
print(pd.DataFrame(m1_boroughs).to_string(index=False))

print("\n=== 3. Disaggregated Borough Comparison (Model 2: WITH Borough) ===")
print(pd.DataFrame(m2_boroughs).to_string(index=False))

=== 1. Overall Comparison Across Fixed Review Capacities ===
         Flagged (M1 No Borough) Precision (M1 No Borough) Recall (M1 No Borough) FPR (M1 No Borough) Flagged (M2 With Borough) Precision (M2 With Borough) Recall (M2 With Borough) FPR (M2 With Borough)
Capacity                                                                                                                                                                                                  
Top 10%                    8,164                     26.2%                  17.5%                8.7%                     8,164                       32.3%                    21.6%                  8.0%
Top 20%                   16,328                     24.1%                  32.3%               17.8%                    16,328                       28.2%                    37.8%                 16.9%
Top 30%                   24,492                     21.7%                  43.7%               27.6%                    24,492

## Section 11: Final Test Evaluation (Frozen Model 1 @ Score Cutoff 0.28)

We unseal the held-out Test Partition (March 1, 2024 – May 31, 2024; $N = 47,825$) and run the final evaluation once using:
- **Frozen Model**: Model 1 (Logistic Regression without `borough`, trained on Oct–Dec 2023).
- **Frozen Operating Cutoff**: Ranking score $\ge 0.28$.

Scores are evaluated as ranking scores, not calibrated probabilities. Conclusions are strictly limited to this single-season chronological split.

In [12]:
# 1. Load the held-out Test Partition from verified artifacts
df_feat_all = pd.read_json(FEATURES_PATH)
df_lab_all = pd.read_json(LABELS_PATH)

test_mask = (df_feat_all["split"] == "test") & (df_lab_all["eligible_for_modeling"])
assert test_mask.sum() == 47825, f"Expected 47,825 eligible test records, got {test_mask.sum()}"

df_features_test = df_feat_all.loc[test_mask].reset_index(drop=True)
df_labels_test = df_lab_all.loc[test_mask].reset_index(drop=True)

# Free memory of full tables
del df_feat_all, df_lab_all

# 2. Extract Features and Labels
X_test_raw = df_features_test[MODEL_FEATURE_COLS].copy()
for col in CATEGORICAL_FEATURES:
    X_test_raw[col] = X_test_raw[col].fillna("MISSING").astype(str)

y_test = df_labels_test["slow_resolution_label"].astype(int).values
boroughs_test = df_features_test["borough"].values

# 3. Transform with Frozen Preprocessor and Score with Frozen Model 1
X_test = preprocessor.transform(X_test_raw)
y_test_scores = lr_model.predict_proba(X_test)[:, 1]

FROZEN_SCORE_CUTOFF = 0.28
y_test_pred = (y_test_scores >= FROZEN_SCORE_CUTOFF).astype(int)

# 4. Overall Test Set Performance
test_base_rate = y_test.mean()
test_roc = roc_auc_score(y_test, y_test_scores)
test_pr = average_precision_score(y_test, y_test_scores)

test_flagged = y_test_pred.sum()
test_share_flagged = test_flagged / len(y_test)

tp_test = ((y_test == 1) & (y_test_pred == 1)).sum()
fp_test = ((y_test == 0) & (y_test_pred == 1)).sum()
tn_test = ((y_test == 0) & (y_test_pred == 0)).sum()
fn_test = ((y_test == 1) & (y_test_pred == 0)).sum()

test_prec = tp_test / (tp_test + fp_test) if (tp_test + fp_test) > 0 else 0.0
test_rec = tp_test / (tp_test + fn_test) if (tp_test + fn_test) > 0 else 0.0
test_fpr = fp_test / (fp_test + tn_test) if (fp_test + tn_test) > 0 else 0.0

print("=================================================================")
print("=== FINAL TEST EVALUATION: FROZEN MODEL 1 (NO BOROUGH) ===")
print("=== Window: March 1, 2024 – May 31, 2024 (Spring Taper) ===")
print("=================================================================\n")

print(f"Total Eligible Test Records : {len(y_test):,}")
print(f"Actual Slow Prevalence      : {test_base_rate:.1%} ({y_test.sum():,} slow tickets)")
print(f"Test ROC-AUC                : {test_roc:.4f}")
print(f"Test PR-AUC (Avg Precision) : {test_pr:.4f} (Benchmark: {test_base_rate:.4f})")
print(f"\nOperating Point (Score Cutoff >= {FROZEN_SCORE_CUTOFF:.2f}):")
print(f"  Total Flagged Requests    : {test_flagged:,} ({test_share_flagged:.1%} of test intake)")
print(f"  Precision                 : {test_prec:.1%}")
print(f"  Recall (Slow Cases Caught): {test_rec:.1%}")
print(f"  False Positive Rate (FPR) : {test_fpr:.1%}")

# 5. Disaggregated Test Performance by Borough
borough_test_rows = []
for b in sorted(np.unique(boroughs_test)):
    m_b = boroughs_test == b
    sub_y = y_test[m_b]
    sub_scores = y_test_scores[m_b]
    sub_pred = y_test_pred[m_b]
    n_b = len(sub_y)
    
    b_base = sub_y.mean()
    b_flagged = sub_pred.sum()
    b_share = b_flagged / n_b
    
    tp_b = ((sub_y == 1) & (sub_pred == 1)).sum()
    fp_b = ((sub_y == 0) & (sub_pred == 1)).sum()
    tn_b = ((sub_y == 0) & (sub_pred == 0)).sum()
    fn_b = ((sub_y == 1) & (sub_pred == 0)).sum()
    
    prec_b = tp_b / (tp_b + fp_b) if (tp_b + fp_b) > 0 else 0.0
    rec_b = tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0.0
    fpr_b = fp_b / (fp_b + tn_b) if (fp_b + tn_b) > 0 else 0.0
    
    borough_test_rows.append({
        "Borough": b,
        "Test Cases": f"{n_b:,}",
        "Actual Slow %": f"{b_base:.1%}",
        "Share Flagged": f"{b_share:.1%}",
        "Precision": f"{prec_b:.1%}",
        "Recall": f"{rec_b:.1%}",
        "FPR": f"{fpr_b:.1%}",
    })

print("\n=== Disaggregated Test Performance by Borough (Cutoff >= 0.28) ===")
print(pd.DataFrame(borough_test_rows).to_string(index=False))

=== FINAL TEST EVALUATION: FROZEN MODEL 1 (NO BOROUGH) ===
=== Window: March 1, 2024 – May 31, 2024 (Spring Taper) ===

Total Eligible Test Records : 47,825
Actual Slow Prevalence      : 9.0% (4,302 slow tickets)
Test ROC-AUC                : 0.6139
Test PR-AUC (Avg Precision) : 0.1397 (Benchmark: 0.0900)

Operating Point (Score Cutoff >= 0.28):
  Total Flagged Requests    : 19,784 (41.4% of test intake)
  Precision                 : 12.3%
  Recall (Slow Cases Caught): 56.4%
  False Positive Rate (FPR) : 39.9%

=== Disaggregated Test Performance by Borough (Cutoff >= 0.28) ===
      Borough Test Cases Actual Slow % Share Flagged Precision Recall   FPR
        BRONX     17,286          7.8%         41.4%     12.2%  64.7% 39.5%
     BROOKLYN     11,820         10.3%         41.4%     15.3%  61.3% 39.1%
    MANHATTAN     11,443         11.2%         40.3%     12.5%  44.7% 39.7%
       QUEENS      6,785          4.8%         42.9%      5.6%  50.5% 42.6%
STATEN ISLAND        491         25.

## Section 12: Final Test Evaluation Findings & Bounded Conclusions

**Final Test Results (Window: March 1, 2024 – May 31, 2024; $N = 47,825$)**:
- **Baseline Prior**: 9.0% slow prevalence (4,302 slow tickets).
- **Discrimination**:
  - Test ROC-AUC: **`0.6139`** (vs 0.5000 random baseline).
  - Test PR-AUC: **`0.1397`** (vs 0.0900 random baseline; +55.2% relative gain).
- **Operating Performance at Frozen Cutoff (score >= 0.28)**:
  - Total Flagged Requests: **19,784 (41.4% of test intake)**.
  - Precision: **12.3%** (vs 9.0% baseline).
  - Recall: **56.4%** (2,427 of 4,302 slow cases caught).
  - False Positive Rate: **39.9%** (17,357 of 43,523 on-time cases flagged).

**Disaggregated Breakdown by Borough (Cutoff >= 0.28)**:
- **Bronx**: 17,286 cases, 7.8% actual slow, 41.4% flagged, 12.2% precision, 64.7% recall, 39.5% FPR.
- **Brooklyn**: 11,820 cases, 10.3% actual slow, 41.4% flagged, 15.3% precision, 61.3% recall, 39.1% FPR.
- **Manhattan**: 11,443 cases, 11.2% actual slow, 40.3% flagged, 12.5% precision, 44.7% recall, 39.7% FPR.
- **Queens**: 6,785 cases, 4.8% actual slow, 42.9% flagged, 5.6% precision, 50.5% recall, 42.6% FPR.
- **Staten Island**: 491 cases, 25.9% actual slow, 42.0% flagged, 33.5% precision, 54.3% recall, 37.6% FPR.

**Critical Methodological Observations & Caveats**:
1. **Workload Shift**:
   - Probability calibration was not established; the fixed 0.28 cutoff did not maintain the intended review volume across the validation and test periods (flagging 19,784 requests, or 41.4% of test intake, compared to ~20% in validation).
2. **Borough Disparities & Non-Causal Framing**:
   - Model 1 exhibited near-uniform selection rates (~40–43%) and uniform False Positive Rates (~38–43%) across all five boroughs.
   - However, precision varied substantially across boroughs (from 5.6% in Queens to 33.5% in Staten Island).
   - **Cautious Non-Causal Framing**: This analysis does not establish the cause of these precision differences or rule out proxy effects or unobserved operational factors (e.g. borough-specific landlord responsiveness, building stock characteristics, or inspection resource allocation).
3. **Generalization Boundaries**:
   - This experiment establishes temporal ranking signal strictly across later chronological periods **within the single 2023–2024 heating season**.
   - It does **not** establish that the model generalizes across multiple seasons or years, nor does it establish that the model is fair.
4. **Final Evaluation Complete**:
   - The one-time final evaluation is complete and the set must not be used for further tuning. No further tuning, feature selection, or cutoff adjustment is performed on these test results.


In [13]:
print("preprocessor in memory:", "preprocessor" in globals())
print("lr_model in memory    :", "lr_model" in globals())

if "lr_model" in globals():
    print("lr_model classes      :", getattr(lr_model, "classes_", None))

preprocessor in memory: True
lr_model in memory    : True
lr_model classes      : [0 1]


In [14]:
import hashlib
import json
from pathlib import Path
import joblib
from sklearn.pipeline import Pipeline

# 1. Ensure output directory exists
model_dir = Path("../../data/models")
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "intake_risk_model_v1.joblib"
meta_path = model_dir / "intake_risk_model_v1.meta.json"

# 2. Package preprocessor and lr_model into a single pipeline
frozen_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", lr_model)
])

# 3. Save the pipeline
joblib.dump(frozen_pipeline, model_path)

# 4. Compute SHA-256 hash for integrity verification
sha256 = hashlib.sha256(model_path.read_bytes()).hexdigest()

# 5. Save metadata sidecar
metadata = {
    "model_name": "frozen_model_1_no_borough",
    "algorithm": "LogisticRegression",
    "operating_cutoff": 0.28,
    "sha256": sha256,
    "features_excluded": ["borough"],
    "calibrated": False,
    "notes": "Score is an uncalibrated ranking score. Evaluated on 2023-2024 cohort only."
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Model saved successfully to: {model_path}")
print(f"SHA-256: {sha256}")
print(f"Metadata saved to: {meta_path}")

Model saved successfully to: ..\..\data\models\intake_risk_model_v1.joblib
SHA-256: 9daffe1971b15fd95c45f9b2fabe5c685bad909d84bfd31459cb811d7ae3b6e2
Metadata saved to: ..\..\data\models\intake_risk_model_v1.meta.json


In [15]:
import json
from pathlib import Path

# Extract 5 rows from X_val_raw (already in memory from validation split only)
sample_val_features = X_val_raw.head(5).to_dict(orient="records")
# Score using in-memory model
sample_scores = lr_model.predict_proba(preprocessor.transform(X_val_raw.head(5)))[:, 1].tolist()

fixture_dir = Path("../../tests/fixtures")
fixture_dir.mkdir(parents=True, exist_ok=True)
fixture_path = fixture_dir / "val_reference_sample.json"

fixture_data = [
    {"features": feat, "expected_score": score}
    for feat, score in zip(sample_val_features, sample_scores)
]

with open(fixture_path, "w", encoding="utf-8") as f:
    json.dump(fixture_data, f, indent=2)

print(f"Validation reference fixture saved with {len(fixture_data)} records to {fixture_path}")

Validation reference fixture saved with 5 records to ..\..\tests\fixtures\val_reference_sample.json
